# Preprocessing: Build Matchup Training Data

This notebook builds the training dataset for the pitcher-batter matchup model.

**Each row = one plate appearance** with:
1. Pitcher historical profile (season stats + per-pitch-type arsenal)
2. Pitcher rolling stats (by last 5/10/20 starts)
3. Batter historical profile (season stats + vs-pitch-type performance)
4. Batter rolling stats (by last 25/50/100 games)
5. Handedness encoding
6. Target outcome (K, BB, 1B, 2B, 3B, HR, OUT)

**Rolling stats include:** whiff%, csw%, K%, BB%, zone%, chase%, xwOBA, exit velo, barrel%, hard hit%

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np

from src.data.preprocess import (
    MatchupPreprocessor,
    prepare_temporal_split,
    OUTCOME_CLASSES,
)

# Import config
from src.config import (
    DATA_START,
    DATA_END,
    VAL_DATE,
    TEST_DATE,
    DATA_DIR,
    PITCHER_ROLLING_WINDOWS,
    BATTER_ROLLING_WINDOWS,
)

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 200)

print(f"Date range: {DATA_START} to {DATA_END}")
print(f"Val date: {VAL_DATE}, Test date: {TEST_DATE}")
print(f"Pitcher rolling windows: {PITCHER_ROLLING_WINDOWS}")
print(f"Batter rolling windows: {BATTER_ROLLING_WINDOWS}")

Date range: 2021-04-01 to 2026-04-01
Val date: 2025-06-01, Test date: 2025-07-15
Pitcher rolling windows: [5, 10, 20]
Batter rolling windows: [25, 50, 100]


In [ ]:
# Memory management utilities
import gc
import psutil

def clear_mem():
    """Run garbage collection and print memory usage."""
    gc.collect()
    mem_gb = psutil.Process().memory_info().rss / 1024**3
    print(f"Memory: {mem_gb:.2f} GB")
    return mem_gb

clear_mem()

## 1. Load Raw Data

Load the Statcast pitches and pre-computed profiles from notebook 02.

In [2]:
# Load data from notebooks 01 and 02 outputs
from pathlib import Path

raw_dir = Path(f'../{DATA_DIR}/raw')
profiles_dir = Path(f'../{DATA_DIR}/profiles')

# Check that data exists
if not raw_dir.exists():
    raise FileNotFoundError(f"Run notebook 01 first: {raw_dir}")
if not profiles_dir.exists():
    raise FileNotFoundError(f"Run notebook 02 first: {profiles_dir}")

# Load pitches
print("Loading pitches...")
pitches = pd.read_parquet(raw_dir / 'pitches.parquet')
print(f"  Pitches: {len(pitches):,} rows")

# Load profiles
print("Loading profiles...")
pitcher_profiles = pd.read_csv(profiles_dir / 'pitcher_arsenal.csv')
batter_profiles = pd.read_csv(profiles_dir / 'batter_profiles.csv')
print(f"  Pitcher profiles: {len(pitcher_profiles):,}")
print(f"  Batter profiles: {len(batter_profiles):,}")

clear_mem()

Loading pitches...
  Pitches: 3,860,179 rows
Loading profiles...
  Pitcher profiles: 1,618
  Batter profiles: 1,291


## 2. Build Matchup Data

Combine at-bat features, profiles, and rolling stats into one training dataset.

In [3]:
# Initialize preprocessor
preprocessor = MatchupPreprocessor()

# Build the matchup dataset
matchups = preprocessor.build_matchup_data(
    pitches_df=pitches,
    pitcher_profiles=pitcher_profiles,
    batter_profiles=batter_profiles,
    pitcher_rolling_windows=PITCHER_ROLLING_WINDOWS,
    batter_rolling_windows=BATTER_ROLLING_WINDOWS,
)

print(f"\nMatchup dataset shape: {matchups.shape}")
matchups.head(10)

Building matchup training data...
  Extracting plate appearances...
  Computing pitcher rolling stats...
  Computing batter rolling stats...
  Merging features...
  Adding handedness features...
  Built 987,069 matchup rows

Matchup dataset shape: (987069, 224)


,game_pk,game_date,at_bat_number,pitcher_id,batter_id,p_throws,stand,events,outcome,season,p_pitcher_id,p_pitcher_name,p_p_throws,p_total_pitches,p_primary_pitch,p_ff_usage_pct,p_si_usage_pct,p_si_velo_avg,p_si_spin_avg,p_si_vmov_avg,p_si_hmov_avg,p_si_whiff_pct,p_fc_usage_pct,p_sl_usage_pct,p_sl_velo_avg,p_sl_spin_avg,p_sl_vmov_avg,p_sl_hmov_avg,p_sl_whiff_pct,p_cu_usage_pct,p_ch_usage_pct,p_sv_usage_pct,p_fs_usage_pct,p_kc_usage_pct,p_st_usage_pct,p_kn_usage_pct,p_whiff_pct,p_swstr_pct,p_csw_pct,p_zone_pct,...,p_roll20_hard_hit_rate,b_roll25_whiff_rate,b_roll25_contact_rate,b_roll25_k_rate,b_roll25_bb_rate,b_roll25_zone_swing_rate,b_roll25_chase_rate,b_roll25_xwoba,b_roll25_avg_exit_velo,b_roll25_barrel_rate,b_roll25_hard_hit_rate,b_roll50_whiff_rate,b_roll50_contact_rate,b_roll50_k_rate,b_roll50_bb_rate,b_roll50_zone_swing_rate,b_roll50_chase_rate,b_roll50_xwoba,b_roll50_avg_exit_velo,b_roll50_barrel_rate,b_roll50_hard_hit_rate,b_roll100_whiff_rate,b_roll100_contact_rate,b_roll100_k_rate,b_roll100_bb_rate,b_roll100_zone_swing_rate,b_roll100_chase_rate,b_roll100_xwoba,b_roll100_avg_exit_velo,b_roll100_barrel_rate,b_roll100_hard_hit_rate,p_throws_L,p_throws_R,stand_L,stand_R,matchup_LvL,matchup_LvR,matchup_RvL,matchup_RvR,same_hand
0,632169,2021-04-10,1,657277,641658,R,R,walk,BB,2021,657277.0,"Webb, Logan",R,15255.0,SI,0.060374,0.351098,92.396714,1912.813147,0.014337,-1.281441,0.115545,0.024844,0.00000,NaN,NaN,NaN,NaN,NaN,0.000000,0.306719,0.0,0.0,0.0,0.256899,0.0,0.216587,0.101344,0.295969,0.486857,...,0.324262,0.211848,0.788152,0.215278,0.113889,0.673051,0.284369,0.331511,80.289500,0.000000,0.103810,0.161621,0.838379,0.218367,0.130272,0.615221,0.259506,0.295860,80.387392,0.003788,0.126623,0.154996,0.845004,0.192424,0.091919,0.626450,0.241975,0.279748,80.894811,0.001873,0.130338,0,1,0,1,0,0,0,1,1
1,632169,2021-04-10,2,657277,641857,R,L,single,1B,2021,657277.0,"Webb, Logan",R,15255.0,SI,0.060374,0.351098,92.396714,1912.813147,0.014337,-1.281441,0.115545,0.024844,0.00000,NaN,NaN,NaN,NaN,NaN,0.000000,0.306719,0.0,0.0,0.0,0.256899,0.0,0.216587,0.101344,0.295969,0.486857,...,0.324262,0.292816,0.707184,0.236111,0.090278,0.765294,0.260161,0.285944,82.275966,0.012422,0.189959,0.284486,0.715514,0.256122,0.135714,0.657287,0.235258,0.338013,82.814006,0.009740,0.215801,0.297520,0.702480,0.279557,0.098413,0.659698,0.248029,0.295273,82.299042,0.019262,0.210161,0,1,1,0,0,0,1,0,0
2,632169,2021-04-10,3,657277,596115,R,R,field_out,OUT,2021,657277.0,"Webb, Logan",R,15255.0,SI,0.060374,0.351098,92.396714,1912.813147,0.014337,-1.281441,0.115545,0.024844,0.00000,NaN,NaN,NaN,NaN,NaN,0.000000,0.306719,0.0,0.0,0.0,0.256899,0.0,0.216587,0.101344,0.295969,0.486857,...,0.324262,0.203504,0.796496,0.256522,0.101449,0.698273,0.338287,0.295434,84.550102,0.000000,0.190051,0.184759,0.815241,0.301389,0.056944,0.685360,0.309580,0.329623,84.320096,0.011696,0.245530,0.217455,0.782545,0.302891,0.097959,0.680159,0.266036,0.315713,85.338132,0.020817,0.261408,0,1,0,1,0,0,0,1,1
3,632169,2021-04-10,4,657277,453568,R,L,field_out,OUT,2021,657277.0,"Webb, Logan",R,15255.0,SI,0.060374,0.351098,92.396714,1912.813147,0.014337,-1.281441,0.115545,0.024844,0.00000,NaN,NaN,NaN,NaN,NaN,0.000000,0.306719,0.0,0.0,0.0,0.256899,0.0,0.216587,0.101344,0.295969,0.486857,...,0.324262,0.358844,0.641156,0.395833,0.062500,0.406378,0.233609,0.227197,72.242519,0.000000,0.237037,0.369269,0.630731,0.460145,0.043478,0.480814,0.221945,0.151633,72.492724,0.000000,0.161470,0.292975,0.707025,0.344746,0.054348,0.568039,0.290242,0.183777,75.419043,0.017778,0.171587,0,1,1,0,0,0,1,0,0
4,632169,2021-04-10,5,592346,600303,R,L,single,1B,2021,592346.0,"González, Chi Chi",R,2086.0,FF,0.430489,0.087728,91.287978,2195.666667,0.619344,-0.951967,0.098901,0.069990,0.24976,85.55048,2513.216216,0.278196,0.373301,0.180451,0.052733,0.108821,0.0,0.0,0.0,0.000000,0.0,0.142857,0.068073,0.243049,0.523969,...,0.262371,0.201393,0.798607,0.339583,0.053472,0.617949,0.427345,0.200037,80.973065,0.041667,0.248611,0.262573,0.73

In [4]:
# All columns in the matchup dataset
print(f"Total columns: {len(matchups.columns)}")
print("\nColumn groups:")

# Pitcher profile features (season-level)
p_cols = [c for c in matchups.columns if c.startswith('p_') and 'roll' not in c]
print(f"\nPitcher profile features ({len(p_cols)}):  [season-level arsenal stats]")
print(p_cols[:15], '...' if len(p_cols) > 15 else '')

# Pitcher rolling features
p_roll_cols = [c for c in matchups.columns if c.startswith('p_roll')]
print(f"\nPitcher rolling features ({len(p_roll_cols)}):  [by last N starts]")
print(p_roll_cols)

# Batter profile features (season-level)
b_cols = [c for c in matchups.columns if c.startswith('b_') and 'roll' not in c]
print(f"\nBatter profile features ({len(b_cols)}):  [season-level vs pitch types]")
print(b_cols[:15], '...' if len(b_cols) > 15 else '')

# Batter rolling features
b_roll_cols = [c for c in matchups.columns if c.startswith('b_roll')]
print(f"\nBatter rolling features ({len(b_roll_cols)}):  [by last N games]")
print(b_roll_cols)

# Handedness features
hand_cols = [c for c in matchups.columns if 'throw' in c or 'stand' in c or 'matchup' in c or 'hand' in c]
print(f"\nHandedness features ({len(hand_cols)}):")
print(hand_cols)

Total columns: 224

Column groups:

Pitcher profile features (81):  [season-level arsenal stats]
['p_throws', 'p_pitcher_id', 'p_pitcher_name', 'p_p_throws', 'p_total_pitches', 'p_primary_pitch', 'p_ff_usage_pct', 'p_si_usage_pct', 'p_si_velo_avg', 'p_si_spin_avg', 'p_si_vmov_avg', 'p_si_hmov_avg', 'p_si_whiff_pct', 'p_fc_usage_pct', 'p_sl_usage_pct'] ...

Pitcher rolling features (33):  [by last N starts]
['p_roll5_whiff_rate', 'p_roll5_csw_rate', 'p_roll5_k_rate', 'p_roll5_bb_rate', 'p_roll5_zone_rate', 'p_roll5_chase_rate', 'p_roll5_avg_velo', 'p_roll5_xwoba', 'p_roll5_avg_exit_velo', 'p_roll5_barrel_rate', 'p_roll5_hard_hit_rate', 'p_roll10_whiff_rate', 'p_roll10_csw_rate', 'p_roll10_k_rate', 'p_roll10_bb_rate', 'p_roll10_zone_rate', 'p_roll10_chase_rate', 'p_roll10_avg_velo', 'p_roll10_xwoba', 'p_roll10_avg_exit_velo', 'p_roll10_barrel_rate', 'p_roll10_hard_hit_rate', 'p_roll20_whiff_rate', 'p_roll20_csw_rate', 'p_roll20_k_rate', 'p_roll20_bb_rate', 'p_roll20_zone_rate', 'p_roll20

In [5]:
# Outcome distribution
print("Outcome distribution:")
outcome_counts = matchups['outcome'].value_counts()
outcome_pcts = matchups['outcome'].value_counts(normalize=True) * 100

pd.DataFrame({
    'count': outcome_counts,
    'pct': outcome_pcts.round(1)
})

Outcome distribution:


,count,pct
outcome,,
OUT,457672,46.4
K,227921,23.1
1B,141405,14.3
BB,81930,8.3
2B,43555,4.4
HR,30877,3.1
3B,3709,0.4


## 3. Explore Feature Groups

In [6]:
# Pitcher rolling features (by starts)
print("Pitcher rolling features (by last N starts):")
print("Stats: whiff_rate, csw_rate, k_rate, bb_rate, zone_rate, chase_rate,")
print("       avg_velo, xwoba, avg_exit_velo, barrel_rate, hard_hit_rate\n")
matchups[p_roll_cols].describe().round(3)

Pitcher rolling features (by last N starts):
Stats: whiff_rate, csw_rate, k_rate, bb_rate, zone_rate, chase_rate,
       avg_velo, xwoba, avg_exit_velo, barrel_rate, hard_hit_rate



,p_roll5_whiff_rate,p_roll5_csw_rate,p_roll5_k_rate,p_roll5_bb_rate,p_roll5_zone_rate,p_roll5_chase_rate,p_roll5_avg_velo,p_roll5_xwoba,p_roll5_avg_exit_velo,p_roll5_barrel_rate,p_roll5_hard_hit_rate,p_roll10_whiff_rate,p_roll10_csw_rate,p_roll10_k_rate,p_roll10_bb_rate,p_roll10_zone_rate,p_roll10_chase_rate,p_roll10_avg_velo,p_roll10_xwoba,p_roll10_avg_exit_velo,p_roll10_barrel_rate,p_roll10_hard_hit_rate,p_roll20_whiff_rate,p_roll20_csw_rate,p_roll20_k_rate,p_roll20_bb_rate,p_roll20_zone_rate,p_roll20_chase_rate,p_roll20_avg_velo,p_roll20_xwoba,p_roll20_avg_exit_velo,p_roll20_barrel_rate,p_roll20_hard_hit_rate
count,986779.000,983913.000,986809.000,986809.000,983913.000,983786.000,983913.000,975188.000,983782.000,983782.000,983782.000,987040.000,986824.000,987060.000,987060.000,986824.000,986824.000,986824.000,983981.000,986796.000,986796.000,986796.000,987060.000,987060.000,987060.000,987060.000,987060.000,987060.000,987060.000,986725.000,987060.000,987060.000,987060.000
mean,0.236,0.278,0.234,0.079,0.498,0.285,89.043,0.311,82.762,0.015,0.247,0.237,0.278,0.235,0.079,0.498,0.285,89.049,0.311,82.760,0.015,0.247,0.237,0.279,0.236,0.080,0.498,0.285,89.090,0.310,82.753,0.015,0.247
std,0.077,0.050,0.093,0.050,0.056,0.072,3.193,0.073,2.882,0.021,0.081,0.062,0.039,0.074,0.038,0.045,0.055,3.033,0.056,2.179,0.015,0.060,0.052,0.032,0.061,0.030,0.037,0.043,2.815,0.043,1.699,0.010,0.045
min,0.000,0.000,0.000,0.000,0.000,0.000,36.371,0.000,51.000,0.000,0.000,0.000,0.000,0.000,0.000,0.237,0.000,42.107,0.000,62.758,0.000,0.000,0.000,0.118,0.000,0.000,0.288,0.077,53.466,0.040,72.058,0.000,0.043
25%,0.186,0.248,0.173,0.048,0.465,0.243,87.382,0.270,81.201,0.000,0.201,0.195,0.254,0.184,0.054,0.471,0.252,87.421,0.278,81.509,0.004,0.212,0.201,0.258,0.193,0.059,0.474,0.258,87.490,0.284,81.750,0.008,0.219
50%,0.229,0.274,0.224,0.073,0.497,0.282,89.130,0.312,82.790,0.010,0.244,0.231,0.275,0.227,0.075,0.498,0.283,89.143,0.312,82.784,0.013,0.246,0.232,0.275,0.229,0.077,0.498,0.283,89.168,0.312,82.781,0.014,0.245
75%,0.277,0.303,0.283,0.103,0.529,0.321,90.917,0.352,84.357,0.022,0.290,0.272,0.298,0.276,0.099,0.525,0.314,90.876,0.344,84.033,0.020,0.281,0.268,0.296,0.271,0.096,0.521,0.309,90.852,0.337,83.822,0.019,0.274
max,1.000,1.000,1.000,1.000,1.000,1.000,101.008,1.133,110.300,1.000,1.000,1.000,0.750,1.000,1.000,0.889,1.000,100.262,1.133,105.300,0.500,1.000,0.619,0.530,0.638,0.298,0.725,0.569,98.583,0.740,90.876,0.143,0.509


In [7]:
# Sample of pitcher arsenal features
pitcher_sample_cols = [
    'p_ff_velo_avg', 'p_ff_spin_avg', 'p_ff_whiff_pct',
    'p_sl_velo_avg', 'p_sl_whiff_pct',
    'p_whiff_pct', 'p_csw_pct', 'p_zone_pct'
]
available = [c for c in pitcher_sample_cols if c in matchups.columns]
print("Pitcher arsenal features sample:")
matchups[available].describe().round(3)

Pitcher arsenal features sample:


,p_ff_velo_avg,p_ff_spin_avg,p_ff_whiff_pct,p_sl_velo_avg,p_sl_whiff_pct,p_whiff_pct,p_csw_pct,p_zone_pct
count,936070.000,936053.000,936070.000,792660.000,792660.000,978695.000,978695.000,978695.000
mean,93.862,2272.237,0.185,85.187,0.313,0.235,0.276,0.493
std,2.282,140.433,0.051,2.569,0.086,0.042,0.022,0.028
min,83.021,1655.700,0.000,71.428,0.000,0.000,0.063,0.167
25%,92.593,2179.422,0.154,83.691,0.254,0.207,0.261,0.477
50%,93.910,2268.575,0.183,85.510,0.305,0.234,0.275,0.493
75%,95.367,2368.388,0.214,86.955,0.366,0.261,0.289,0.511
max,101.566,2783.559,0.667,94.129,0.714,0.529,0.382,0.679


In [8]:
# Sample of batter profile features
batter_sample_cols = [
    'b_whiff_pct', 'b_contact_pct', 'b_chase_pct',
    'b_vs_ff_whiff_pct', 'b_vs_sl_whiff_pct',
    'b_velo_97_plus_whiff_pct', 'b_avg_exit_velo', 'b_xwoba'
]
available = [c for c in batter_sample_cols if c in matchups.columns]
print("Batter profile features sample:")
matchups[available].describe().round(3)

Batter profile features sample:


,b_whiff_pct,b_contact_pct,b_chase_pct,b_vs_ff_whiff_pct,b_vs_sl_whiff_pct,b_velo_97_plus_whiff_pct,b_avg_exit_velo,b_xwoba
count,976270.000,976270.000,976270.000,975297.000,970779.000,948943.000,970589.000,970589.000
mean,0.233,0.767,0.286,0.188,0.321,0.211,88.476,0.373
std,0.058,0.058,0.057,0.064,0.072,0.070,2.693,0.050
min,0.063,0.392,0.070,0.000,0.000,0.000,55.809,0.128
25%,0.197,0.728,0.246,0.143,0.273,0.161,86.904,0.339
50%,0.233,0.767,0.283,0.186,0.320,0.206,88.631,0.369
75%,0.272,0.803,0.320,0.228,0.370,0.258,90.268,0.406
max,0.608,0.937,0.681,0.595,0.933,0.562,95.963,0.623


In [9]:
# Batter rolling features (by games)
print("Batter rolling features (by last N games):")
print("Stats: whiff_rate, contact_rate, k_rate, bb_rate, zone_swing_rate, chase_rate,")
print("       xwoba, avg_exit_velo, barrel_rate, hard_hit_rate\n")
matchups[b_roll_cols].describe().round(3)

Batter rolling features (by last N games):
Stats: whiff_rate, contact_rate, k_rate, bb_rate, zone_swing_rate, chase_rate,
       xwoba, avg_exit_velo, barrel_rate, hard_hit_rate



,b_roll25_whiff_rate,b_roll25_contact_rate,b_roll25_k_rate,b_roll25_bb_rate,b_roll25_zone_swing_rate,b_roll25_chase_rate,b_roll25_xwoba,b_roll25_avg_exit_velo,b_roll25_barrel_rate,b_roll25_hard_hit_rate,b_roll50_whiff_rate,b_roll50_contact_rate,b_roll50_k_rate,b_roll50_bb_rate,b_roll50_zone_swing_rate,b_roll50_chase_rate,b_roll50_xwoba,b_roll50_avg_exit_velo,b_roll50_barrel_rate,b_roll50_hard_hit_rate,b_roll100_whiff_rate,b_roll100_contact_rate,b_roll100_k_rate,b_roll100_bb_rate,b_roll100_zone_swing_rate,b_roll100_chase_rate,b_roll100_xwoba,b_roll100_avg_exit_velo,b_roll100_barrel_rate,b_roll100_hard_hit_rate
count,987065.000,987065.000,987065.000,987065.000,987053.000,987053.000,982047.000,987018.000,987018.000,987018.000,987065.000,987065.000,987065.000,987065.000,987065.000,987065.000,984846.000,987065.000,987065.000,987065.000,987065.000,987065.000,987065.000,987065.000,987065.000,987065.000,986188.000,987065.000,987065.000,987065.000
mean,0.228,0.772,0.232,0.081,0.676,0.288,0.310,83.103,0.016,0.258,0.228,0.772,0.233,0.081,0.676,0.288,0.310,83.087,0.016,0.258,0.229,0.771,0.234,0.081,0.676,0.288,0.310,83.072,0.016,0.258
std,0.070,0.070,0.079,0.041,0.071,0.074,0.058,2.716,0.015,0.070,0.063,0.063,0.070,0.033,0.063,0.065,0.048,2.305,0.011,0.058,0.058,0.058,0.063,0.028,0.056,0.059,0.040,2.020,0.009,0.050
min,0.000,0.350,0.000,0.000,0.000,0.000,0.000,56.257,0.000,0.000,0.025,0.492,0.000,0.000,0.403,0.057,0.000,64.649,0.000,0.019,0.032,0.554,0.000,0.000,0.440,0.097,0.000,70.341,0.000,0.073
25%,0.179,0.726,0.176,0.051,0.630,0.237,0.274,81.520,0.004,0.212,0.184,0.729,0.183,0.057,0.635,0.243,0.280,81.710,0.007,0.219,0.189,0.731,0.189,0.061,0.638,0.247,0.283,81.803,0.009,0.224
50%,0.225,0.775,0.227,0.076,0.676,0.284,0.310,83.263,0.013,0.258,0.227,0.773,0.231,0.077,0.676,0.285,0.309,83.235,0.014,0.258,0.228,0.772,0.234,0.078,0.676,0.286,0.309,83.203,0.015,0.258
75%,0.274,0.821,0.282,0.106,0.724,0.335,0.346,84.885,0.023,0.304,0.271,0.816,0.278,0.101,0.718,0.330,0.339,84.625,0.022,0.297,0.269,0.811,0.276,0.098,0.713,0.326,0.334,84.440,0.021,0.291
max,0.650,1.000,0.795,0.375,1.000,1.000,0.924,100.100,0.250,0.875,0.508,0.975,0.731,0.257,0.894,1.000,0.853,92.018,0.125,0.526,0.446,0.968,0.567,0.223,0.861,1.000,0.639,92.018,0.062,0.475


In [10]:
# Handedness distribution
print("Handedness matchups:")
matchup_cols = ['matchup_LvL', 'matchup_LvR', 'matchup_RvL', 'matchup_RvR']
available = [c for c in matchup_cols if c in matchups.columns]
matchups[available].sum()

Handedness matchups:


matchup_LvL     77462
matchup_LvR    193899
matchup_RvL    333199
matchup_RvR    382509
dtype: int64

In [11]:
# Check null percentages
null_pcts = (matchups.isnull().sum() / len(matchups) * 100).sort_values(ascending=False)
print("Columns with >50% nulls:")
print(null_pcts[null_pcts > 50])

print(f"\nTotal columns: {len(matchups.columns)}")
print(f"Columns with any nulls: {(null_pcts > 0).sum()}")
print(f"Columns with >50% nulls: {(null_pcts > 50).sum()}")

Columns with >50% nulls:
b_batter_name          100.000000
p_kn_spin_avg           99.822100
p_kn_vmov_avg           99.822100
p_kn_hmov_avg           99.822100
p_kn_velo_avg           99.822100
p_kn_whiff_pct          99.822100
b_vs_kn_xwoba           99.772559
b_vs_kn_contact_pct     99.141904
b_vs_kn_whiff_pct       99.141904
p_sv_velo_avg           97.001324
p_sv_vmov_avg           97.001324
p_sv_spin_avg           97.001324
p_sv_whiff_pct          97.001324
p_sv_hmov_avg           97.001324
p_kc_whiff_pct          87.520528
p_kc_vmov_avg           87.520528
p_kc_hmov_avg           87.520528
p_kc_spin_avg           87.520528
p_kc_velo_avg           87.520528
p_fs_velo_avg           82.356046
p_fs_spin_avg           82.356046
p_fs_vmov_avg           82.356046
p_fs_hmov_avg           82.356046
p_fs_whiff_pct          82.356046
p_st_spin_avg           59.532920
p_st_velo_avg           59.532920
p_st_hmov_avg           59.532920
p_st_whiff_pct          59.532920
p_st_vmov_avg          

## 4. Preprocess for Model

Scale numeric features and encode target.

In [12]:
# Identify feature columns
numeric_cols, binary_cols = preprocessor.get_feature_columns(matchups)

print(f"Numeric features: {len(numeric_cols)}")
print(f"Binary features: {len(binary_cols)}")
print(f"Total features: {len(numeric_cols) + len(binary_cols)}")

print("\nBinary features:")
print(binary_cols)

Numeric features: 198
Binary features: 9
Total features: 207

Binary features:
['p_throws_L', 'p_throws_R', 'stand_L', 'stand_R', 'matchup_LvL', 'matchup_LvR', 'matchup_RvL', 'matchup_RvR', 'same_hand']


In [13]:
# Temporal split (avoid leakage)
# Use last ~2 weeks as test, previous ~2 weeks as validation

train_df, test_df, val_df = prepare_temporal_split(
    matchups,
    test_date=TEST_DATE,
    val_date=VAL_DATE,
)

print(f"Train: {len(train_df):,} rows ({train_df['game_date'].min()} to {train_df['game_date'].max()})")
print(f"Val:   {len(val_df):,} rows ({val_df['game_date'].min()} to {val_df['game_date'].max()})")
print(f"Test:  {len(test_df):,} rows ({test_df['game_date'].min()} to {test_df['game_date'].max()})")

Train: 853,340 rows (2021-04-01 00:00:00 to 2025-05-31 00:00:00)
Val:   42,831 rows (2025-06-01 00:00:00 to 2025-07-13 00:00:00)
Test:  90,898 rows (2025-07-18 00:00:00 to 2026-03-30 00:00:00)


In [14]:
# Fit preprocessor on training data only
preprocessor.fit(train_df)

# Transform all sets
X_train, y_train = preprocessor.transform(train_df, include_target=True)
X_val, y_val = preprocessor.transform(val_df, include_target=True)
X_test, y_test = preprocessor.transform(test_df, include_target=True)

print(f"X_train shape: {X_train.shape}")
print(f"X_val shape: {X_val.shape}")
print(f"X_test shape: {X_test.shape}")

print(f"\nFeature names: {len(preprocessor.get_feature_names())}")
print(f"Outcome classes: {preprocessor.get_outcome_classes()}")

clear_mem()

X_train shape: (853340, 207)
X_val shape: (42831, 207)
X_test shape: (90898, 207)

Feature names: 207
Outcome classes: [np.str_('1B'), np.str_('2B'), np.str_('3B'), np.str_('BB'), np.str_('HR'), np.str_('K'), np.str_('OUT')]


In [15]:
# Check scaled features look reasonable
print("Scaled feature statistics (train):")
print(f"Mean: {np.nanmean(X_train, axis=0)[:5].round(2)}... (first 5)")
print(f"Std:  {np.nanstd(X_train, axis=0)[:5].round(2)}... (first 5)")
print(f"Min:  {np.nanmin(X_train, axis=0)[:5].round(2)}... (first 5)")
print(f"Max:  {np.nanmax(X_train, axis=0)[:5].round(2)}... (first 5)")

# Check NaN preservation
nan_count = np.isnan(X_train).sum()
print(f"\nNaN values preserved: {nan_count:,} ({nan_count / X_train.size * 100:.1f}%)")

Scaled feature statistics (train):
Mean: [0.01 0.01 0.01 0.55 0.54]... (first 5)
Std:  [1.   0.99 1.   0.06 0.16]... (first 5)
Min:  [-1.43 -1.83 -0.95  0.2  -0.05]... (first 5)
Max:  [2.41 3.12 4.07 0.74 1.  ]... (first 5)

NaN values preserved: 30,173,976 (17.1%)


In [16]:
# Target distribution in each split
print("Target distribution (%):\n")
for name, y in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    unique, counts = np.unique(y, return_counts=True)
    pcts = counts / len(y) * 100
    print(f"{name}:")
    for i, pct in enumerate(pcts):
        print(f"  {preprocessor.get_outcome_classes()[i]}: {pct:.1f}%")
    print()

Target distribution (%):

Train:
  1B: 14.3%
  2B: 4.4%
  3B: 0.4%
  BB: 8.3%
  HR: 3.1%
  K: 23.1%
  OUT: 46.3%

Val:
  1B: 14.5%
  2B: 4.3%
  3B: 0.3%
  BB: 7.9%
  HR: 3.2%
  K: 22.2%
  OUT: 47.5%

Test:
  1B: 14.2%
  2B: 4.3%
  3B: 0.3%
  BB: 8.4%
  HR: 3.3%
  K: 23.2%
  OUT: 46.2%



## 5. Save Preprocessed Data

In [17]:
from pathlib import Path
from src.config import MODELS_DIR

# Save matchup DataFrame
output_dir = Path(f'../{DATA_DIR}/processed')
output_dir.mkdir(parents=True, exist_ok=True)

matchups.to_parquet(output_dir / 'matchups.parquet', index=False)
print(f"Saved matchups to {output_dir / 'matchups.parquet'}")

# Save train/val/test DataFrames
train_df.to_parquet(output_dir / 'train.parquet', index=False)
val_df.to_parquet(output_dir / 'val.parquet', index=False)
test_df.to_parquet(output_dir / 'test.parquet', index=False)
print(f"Saved train/val/test splits")

# Save preprocessor
models_dir = Path(f'../{MODELS_DIR}')
models_dir.mkdir(parents=True, exist_ok=True)
preprocessor.save(models_dir / 'matchup_preprocessor.pkl')

clear_mem()

Saved matchups to ../data/processed/matchups.parquet
Saved train/val/test splits
Saved preprocessor to ../models/matchup_preprocessor.pkl


In [18]:
# Optionally save numpy arrays for direct model training
np.save(output_dir / 'X_train.npy', X_train)
np.save(output_dir / 'X_val.npy', X_val)
np.save(output_dir / 'X_test.npy', X_test)
np.save(output_dir / 'y_train.npy', y_train)
np.save(output_dir / 'y_val.npy', y_val)
np.save(output_dir / 'y_test.npy', y_test)

print("Saved numpy arrays for model training")

Saved numpy arrays for model training


## Summary

The matchup dataset is ready for model training with:

**Features:**
- Pitcher arsenal (per pitch type: velo, spin, movement, usage%, whiff%)
- Pitcher rolling (last 5/10/20 starts): whiff%, csw%, K%, BB%, zone%, chase%, xwOBA, EV, barrel%, hard_hit%
- Batter profile (vs pitch types, velocity buckets, movement profiles)
- Batter rolling (last 25/50/100 games): whiff%, contact%, K%, BB%, zone_swing%, chase%, xwOBA, EV, barrel%, hard_hit%
- Handedness encoding (LvL, LvR, RvL, RvR + individual L/R flags)

**Target:** 7-class outcome (K, BB, 1B, 2B, 3B, HR, OUT)

**Preprocessing:**
- Numeric features scaled (StandardScaler)
- NaN values preserved (for tree models)
- Temporal split (no future leakage)